## Test Optimization Functions

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import plotly.graph_objects as go
import matplotlib.pyplot as plt
import pylupnt as pnt

### 1. Problem Config Setup

In [ ]:
from src.constellation_design import (
    setup_problem_config,
    setup_hybrid_walker_constellation,
)
from src.failure_model import NavSatFailureModel

objs = ["dop", "nsat"]
et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

# satellite fail model
fmodel_type = "short"  # "short" or "long"
if fmodel_type == "short":
    failmodel = NavSatFailureModel(
        tau1=0.25,
        S_tau1=0.95,
        beta0=0.6,
        lam1=0.008,
        tau2=5.0,
        targets_years=(6.0, 8.0),
        targets_survival=(0.8, 0.5),
    )
    max_fail = 2
elif fmodel_type == "long":
    failmodel = NavSatFailureModel(
        tau1=0.25,
        S_tau1=0.99,
        beta0=0.6,
        lam1=0.008,
        tau2=10.0,
        targets_years=(12.0, 15.0),
        targets_survival=(0.9, 0.75),
    )
    max_fail = 1

# config
config = setup_problem_config(
    objs,
    n_walker=3,
    n_phase=3,
    sat_range_phases=[[4, 10], [10, 30], [20, 50]],
    float_sma=False,
    n_users=50,
    elev_mask_deg=5.0,
    cn0_thresh_user=30.0,
    dop_type_phase=["hdop", "pdop", "pdop"],
    use_wdop=True,
    dop_target=[20.0, 15.0, 10.0],
    compute_link_budget=True,
    lat_masks=[[-90, -70], [-90, -30], [-90, 90]],
    launch_years=[0, 5, 10],
    eval_years=[1, 6, 11],
    fail_model=failmodel,
    max_fail_sat=max_fail,
    epoch=et0,
    dt=30 * 60,
)

print("num var:", config["n_var"])
print("  num walker:", config["n_walker"])
print("  num phase:", config["n_phase"])
print("xl=", config["xl"])
print("xu=", config["xu"])

In [ ]:
from src.constellation_design import create_x
from src.init_population import allocate_sats_to_stage

# walker parameters
# 0: semi-major axis [km]
# 1: eccentricity
# 2: sign of argument of periapsis [1 or 0(=-1)]
# 3: number of planes
# 4: number of satellites per plane
# 5: relative phase
# 6: Omega0: right ascension of the ascending node [rad]
walker_params = [
    np.array([2, 0.6383, 1, 3, 2, 1, 0]),  # phase 0
    np.array([2, 0.001, 0, 4, 4, 1, 0]),  # phase 1
    np.array([2, 0.5, 0, 2, 3, 1, 0]),  # phase 2
]
x_phase = []

walker_sats = np.array([6, 16, 6])  # number of satellites in each walker phase (28)
phase_sats = np.array([6, 6, 16])  # number of satellites in each phase
preference_walker = np.array([0, 1, 2])  # preference for walker phases
x_phase, success = allocate_sats_to_stage(
    walker_sats, phase_sats, preference_walker, debug=True
)

x0 = create_x(walker_params, x_phase)

In [ ]:
from src.constellation_design import print_lunanet_x

print_lunanet_x(x0, config)

In [ ]:
from src.postprocess import plot_user_positions

plot_user_positions(config["x_user"], lat_range=(-90, -70))

In [ ]:
# plot fail model
from src.postprocess import plot_survive_rates

plot_survive_rates(
    config["fail_model"],
    config["launch_years"],
    config["eval_years"],
    age_max=15,
    figname="figs/fail_model.pdf",
)

## 2. Objective Computation

In [ ]:
from src.constellation_design import get_sim_time

n_walker = config["n_walker"]
n_phase = config["n_phase"]
tspan = config["tspan"]
et0 = config["tai0"]
x_user = config["x_user"]
objs = config["objs"]
float_sma = config["float_sma"]
err = False

tsim = get_sim_time(x0, config["n_walker"])
tspan = np.linspace(0, tsim, int(tsim / config["dt"]) + 1)  # [s] time since first epoch

print("sim time: {:.1f} days".format(tsim / pnt.SECS_DAY))

t_tai = et0 + tspan  # [s] time since first epoch in TAI
print("float_sma:", float_sma)

In [ ]:
dyn = pnt.NBodyDynamics()
dyn.set_integrator(pnt.IntegratorType.RKF45)
dyn.set_integrator_params(pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10))
dyn.add_body(pnt.Body.Moon(2, 2))
dyn.add_body(pnt.Body.Earth())
dyn.add_body(pnt.Body.Sun())
dyn.set_time_step(60)
dyn.set_frame(pnt.MOON_CI)

In [ ]:
from src.constellation_design import setup_hybrid_walker_constellation

x_orb, x_phase, lunanet_antennas, params_walker = setup_hybrid_walker_constellation(
    x0,
    et0 + tspan,
    dyn,
    float_sma,
    n_walker,
    n_phase,
    compute_link_budget=True,
    debug=True,
)
nsat_used = x_orb.shape[0]
x_orb_mci = params_walker["x_orb_mci"]

In [ ]:
# plot constellation
fig = go.Figure()
pnt.plot.plot_orbits(fig, x_orb)
pnt.plot.plot_body(
    fig,
    pnt.MOON,
    size_factor=2,
    alpha=0.5,
)
pnt.plot.set_view(fig, -80, 20, 2.5)
fig.update_layout(showlegend=True, width=400, height=400)
fig.show()

In [ ]:
from src.constellation_design import compute_coverage_dop

# test objective computation
coverage_phases, dop_phases, ures_noise, od_err, prob_phases = compute_coverage_dop(
    t_tai,
    x_orb,
    x_phase,
    lunanet_antennas=lunanet_antennas,
    coe=params_walker["coes"],
    config=config,
    debug=True,
)

In [ ]:
from src.postprocess import plot_coverage_dop

plot_coverage_dop(
    coverage_phases,
    dop_phases,
    prob_phases,
    x_user,
    config["users_used"],
    tspan,
    dop_thresholds=[20, 15, 8],
    perctile=50,
)

In [ ]:
from src.postprocess import plot_ures

ures_od = np.tile(od_err, (ures_noise.shape[0], 1, 1))
ures = ures_noise + ures_od

plot_ures(ures, x_user, tspan, n_plot=6)

In [ ]:
# constraints function
from src.constellation_design import compute_constraints

print("constraints number: ", config["n_ieq_constr"])

consts = compute_constraints(x0, config)

print("constraints:", consts)

## Test Computation Time for Objective Evaluations

In [ ]:
from src.constellation_design import compute_objectives
import time

## Computation speed chech
objs = ["dop", "nsat"]
n_users = [200, 100, 50]
dts = [5 * 60, 10 * 60, 30 * 60]  # [s]
float_sma = False
n_walker = 3
n_phase = 3

max_fail = 1  # max number of simultaneous satellite failures to consider in OD sim
first = True

for nu in n_users:
    for dt in dts:
        print(f"\n--- n_users={nu}, dt={dt/60} min ---")
        tstart = time.time()
        config = setup_problem_config(
            objs,
            n_walker=n_walker,
            n_phase=n_phase,
            sat_range_phases=[[4, 10], [10, 30], [20, 50]],
            float_sma=float_sma,
            n_users=nu,
            elev_mask_deg=5.0,
            cn0_thresh_user=30.0,
            dop_type_phase=["hdop", "pdop", "pdop"],
            use_wdop=True,
            dop_target=[20.0, 15.0, 8.0],
            compute_link_budget=True,
            lat_masks=[[-90, -70], [-90, -30], [-90, 90]],
            launch_years=[0, 5, 10],
            eval_years=[1, 6, 11],
            fail_model=failmodel,
            max_fail_sat=max_fail,
            epoch=et0,
            dt=dt,
        )
        print("  setup time: {:.2f} s".format(time.time() - tstart))

        tstart = time.time()
        x_orb, x_phase, lunanet_antennas, params_walker = (
            setup_hybrid_walker_constellation(
                x0,
                et0 + tspan,
                dyn,
                float_sma,
                n_walker,
                n_phase,
                compute_link_budget=True,
                debug=False,
            )
        )
        tend = time.time()
        print(f"  setup constellation time (raw): {tend - tstart:.2f} s")

        tstart = time.time()
        objvals, _, _ = compute_objectives(x0, config, debug=True)
        objvals_dop = objvals[0::2]
        if first:  # use the first one as reference
            objvals_ref = objvals_dop
            first = False
            rel_err = np.zeros_like(objvals_dop)
        else:
            rel_err = np.abs((objvals_dop - objvals_ref) / objvals_ref)
            print("  rel err in dop obj: ", rel_err)

        tend = time.time()
        print(
            f"  obj sim time: {tend - tstart:.2f} s   objs: {objvals_dop}   rel err: {rel_err}"
        )